# Fishery (Overshoot and Collapse)

*System Dynamics Studio, model library: Complex behavior*

Fish regenerate logistically. The fishing fleet grows by reinvesting part of its catch and shrinks as boats are retired. Because the fleet responds to past catches, it keeps growing after the fish stock has started to fall: the fishery overshoots its sustainable yield and the catch collapses before the fleet adjusts.

**Concepts:** Overshoot and collapse, Renewable resource, Policy design

**Expected behavior:** The fleet grows on early catches, harvest overshoots the maximum sustainable yield (125 tons/year) and the fish stock collapses before the fleet shrinks.

**Limitations:** Fish price and costs are constant, and the fleet responds only to the catch.

## Model structure

**Stock balances**

$$
\begin{aligned}
\frac{d\,\text{Fish}}{dt} &= \text{regeneration} - \text{harvest} \\
\frac{d\,\text{Boats}}{dt} &= \text{boat\_purchases} - \text{boat\_retirement} \\
\end{aligned}
$$

**Stocks**

| Stock | Initial value | Unit |
|---|---|---|
| `Fish` | 900 | tons |
| `Boats` | 5 | boats |

**Flows and auxiliaries** (in evaluation order)

$$
\begin{aligned}
\text{regeneration} &= \text{Fish}\cdot\text{regeneration\_rate}\cdot\left(1-\frac{\text{Fish}}{\text{carrying\_capacity}}\right) \\
\text{harvest} &= \frac{\text{Boats}\cdot\text{catch\_per\_boat}\cdot\text{Fish}}{\text{carrying\_capacity}} \\
\text{boat\_purchases} &= \frac{\text{harvest}\cdot\text{reinvestment\_fraction}}{\text{boat\_price}} \\
\text{boat\_retirement} &= \frac{\text{Boats}}{\text{boat\_lifetime}} \\
\end{aligned}
$$

**Parameters**

| Parameter | Value | Unit |
|---|---|---|
| `regeneration_rate` | 0.5 | 1/year |
| `carrying_capacity` | 1000 | tons |
| `catch_per_boat` | 20 | tons/(boat*year) |
| `reinvestment_fraction` | 0.15 | dimensionless |
| `boat_price` | 5 | tons/boat |
| `boat_lifetime` | 10 | year |

## Setup

NumPy, SciPy and Matplotlib are preinstalled on Google Colab. The step, ramp and pulse helpers
are the time functions available in the app; they are defined here for completeness.

In [ ]:
# Generated by System Dynamics Studio
# Model time: 0 to 60 year, output every 0.1
# In-app solver: RK4, fixed dt = 0.1.
# This script integrates with scipy.integrate.odeint (adaptive LSODA), so values
# match the app within numerical tolerance, not digit for digit.
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# Time functions (discontinuous; same definitions as in the app)
def sd_step(height, start, t):
    return np.where(t >= start, height, 0.0)

def sd_ramp(slope, start, *rest):
    *end, t = rest
    until = np.minimum(t, end[0]) if end else t
    return np.where(t > start, slope * (until - start), 0.0)

def sd_pulse(height, start, width, t):
    return np.where((t >= start) & (t < start + width), height, 0.0)

## Parameters and initial values

Edit the values below and rerun the notebook from here to try your own scenarios.

In [ ]:
# 1. Model Parameters (Constants)
params = {
    'regeneration_rate': 0.5,  # 1/year
    'carrying_capacity': 1000,  # tons
    'catch_per_boat': 20,  # tons/(boat*year)
    'reinvestment_fraction': 0.15,  # dimensionless
    'boat_price': 5,  # tons/boat
    'boat_lifetime': 10  # year
}

# 2. Initial State (Stocks)
y0 = [
    900,  # Fish [tons]
    5  # Boats [boats]
]

## Model equations

`model` returns the rate of change of every stock. It is the system of ordinary differential equations shown above.

In [ ]:
# 3. Model Definition
def model(y, t, p):
    # Unpack stocks
    Fish, Boats = y

    # Auxiliaries & Flows (dependency order)
    regeneration = ((Fish * p['regeneration_rate']) * ((1 - (Fish / p['carrying_capacity']))))
    harvest = (((Boats * p['catch_per_boat']) * Fish) / p['carrying_capacity'])
    boat_purchases = ((harvest * p['reinvestment_fraction']) / p['boat_price'])
    boat_retirement = (Boats / p['boat_lifetime'])

    # Derivatives (Change in Stocks)
    dFish_dt = regeneration - harvest
    dBoats_dt = boat_purchases - boat_retirement

    return [dFish_dt, dBoats_dt]

## Simulation

The model runs from t = 0 to t = 60 years. The app integrates with RK4 at a fixed
dt = 0.1; this notebook uses SciPy's adaptive `odeint` (LSODA) with tight tolerances, so results agree
with the app within numerical tolerance.

In [ ]:
# 4. Run Simulation
t = np.linspace(0, 60, 601)
sol = odeint(model, y0, t, args=(params,), rtol=1e-8, atol=1e-10)

In [ ]:
# 5. Post-Processing (Calculate variables for plotting)
Fish = sol[:, 0]
Boats = sol[:, 1]
regeneration_rate = np.full_like(t, params['regeneration_rate'])
carrying_capacity = np.full_like(t, params['carrying_capacity'])
catch_per_boat = np.full_like(t, params['catch_per_boat'])
reinvestment_fraction = np.full_like(t, params['reinvestment_fraction'])
boat_price = np.full_like(t, params['boat_price'])
boat_lifetime = np.full_like(t, params['boat_lifetime'])
regeneration = ((Fish * params['regeneration_rate']) * ((1 - (Fish / params['carrying_capacity'])))) + np.zeros_like(t)
harvest = (((Boats * params['catch_per_boat']) * Fish) / params['carrying_capacity']) + np.zeros_like(t)
boat_purchases = ((harvest * params['reinvestment_fraction']) / params['boat_price']) + np.zeros_like(t)
boat_retirement = (Boats / params['boat_lifetime']) + np.zeros_like(t)

## Results

In [ ]:
# 6. Visualization
plt.figure(figsize=(10, 6))
plt.plot(t, Fish, label='Fish', linewidth=2)
plt.plot(t, Boats, label='Boats', linewidth=2)
plt.plot(t, harvest, label='harvest', linewidth=2)

plt.title('System Dynamics Simulation (Python)')
plt.xlabel('Time (year)')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Experiment

Reinvesting less of the revenue in boats: does the fishery stay productive?

The cell below reruns the model with one change and compares each stock against the baseline.

In [ ]:
STOCKS = ['Fish', 'Boats']

def simulate(param_changes=None, initial_changes=None):
    """Rerun the model with changed parameters and/or initial stock values."""
    p = {**params, **(param_changes or {})}
    y_init = list(y0)
    for stock, value in (initial_changes or {}).items():
        y_init[STOCKS.index(stock)] = value
    return odeint(model, y_init, t, args=(p,), rtol=1e-8, atol=1e-10)

scenario = simulate(param_changes={'reinvestment_fraction': 0.05}, initial_changes={})

fig, axes = plt.subplots(len(STOCKS), 1, sharex=True, figsize=(10, 2.8 * len(STOCKS)))
axes = np.atleast_1d(axes)
for i, (ax, stock) in enumerate(zip(axes, STOCKS)):
    ax.plot(t, sol[:, i], label='Baseline', linewidth=2)
    ax.plot(t, scenario[:, i], '--', label='reinvestment_fraction = 0.05', linewidth=2)
    ax.set_ylabel(stock)
    ax.grid(True, alpha=0.3)
axes[0].legend(loc='best')
axes[-1].set_xlabel('Time (year)')
fig.suptitle('Baseline vs. scenario')
plt.tight_layout()
plt.show()

## Questions to explore

- What is the highest harvest the fish can sustain? Where is it reached?
- Which policy keeps the fishery productive: fewer boats, shorter boat life, or lower reinvestment?